# Graph-A2 Table 1 复现与 v11 实验进展报告

**项目**：One-Shot Data Selection for Medical Image Classification via Graph Coverage 复现与 v11 扩展  
**数据集**：OrganSMNIST、OrganAMNIST、PathMNIST、TissueMNIST、BloodMNIST  
**代码分支**：`codex/table1-v11-execution`  
**本报告基线提交**：`eca0bac`  
**静态快照日期**：2026-08-27

> 本 notebook 同时包含已确认结果的静态快照和从 HPC 输出目录动态加载结果的代码。在 `/project/prj-sis01/xuxiaoyu/graph_bench` 中运行全部单元格，可自动更新尚在运行的实验。

## 1. 当前结论摘要

1. 作者 Graph-A2 的十组冻结子集已经全部生成：5 个数据集 × 2 个比例。
2. Graph-A2 seed 42 下游 test 已完成 10/10；主结果使用 best-epoch BA，final epoch 仅作诊断。TissueMNIST 的 best BA 接近论文。
3. v11 selection 已完成 100/100：十组 `a0_original` 直接逐字节复用 Table 1 冻结 indices，另外生成 90 组候选子集。
4. v11 seed-42、5% validation 已完成 25/25。按 best BA +0.5 pp 和 final worst-recall -2 pp 的初步门槛，OrganA、Tissue 有正向候选；OrganS、Path、Blood 暂时保留 A0。
5. 三种子 validation、Graph-A2 三种子 test、winner freeze、最终五种子 test 已组成 Slurm 依赖链。
6. v11 默认只允许 validation。只有 CLI 显式传入 `--evaluation-split test` 才可读取 test，并生成 `test_result.json`。

In [ ]:
from pathlib import Path
import json
import subprocess
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'table1_reproduction').is_dir() and (candidate / 'v11').is_dir():
            return candidate.resolve()
    return Path('/project/prj-sis01/xuxiaoyu/graph_bench')

PROJECT = find_project_root()
A2_ROOT = PROJECT / 'table1_reproduction/outputs/job2_table1'
V11_SELECTION_ROOT = PROJECT / 'v11/outputs/job1_table1_calibration'
V11_VAL_ROOT = PROJECT / 'v11/outputs/job2_table1_validation'
FROZEN_ROOT = PROJECT / 'v11/outputs/frozen_protocol'
V11_TEST_ROOT = PROJECT / 'v11/outputs/job2_table1_test'

def read_result_jsons(root, pattern='*_result.json'):
    rows = []
    if root.exists():
        for path in sorted(root.rglob(pattern)):
            row = json.loads(path.read_text(encoding='utf-8'))
            row['result_path'] = str(path)
            rows.append(row)
    return pd.DataFrame(rows)

print('PROJECT =', PROJECT)
print('Live HPC outputs available =', A2_ROOT.exists() and V11_VAL_ROOT.exists())

## 2. 固定实验协议

| 项目 | 固定设置 |
|---|---|
| 图像 | MedMNIST+ 224×224 |
| 表征 | UNI ViT-L/16，1024 维 embedding |
| Graph-A2 | global k-NN，k=10，H=2，K=A+A² |
| 比例 | 2%、5%，按类别等额预算 |
| 下游模型 | ResNet-18，从零训练，`pretrained=False` |
| 优化器 | SGD，lr=0.1，momentum=0.9，weight decay=5e-4 |
| 训练 | 1000 epochs，batch size=256，cosine LR，无 augmentation |
| 训练种子 | validation: 42/43/44；最终 test: 42/43/44/45/46 |
| 主要指标 | best-epoch balanced accuracy；final BA、final worst-class recall、final CVaR20 仅作诊断 |

Table 1 复现固定使用作者提交 `8cf757adc4c333dc1427d511f0de2f246d15ebac` 的隔离 vendor。v11 修改不会改变该对照实现。

In [ ]:
paper_rows = [
    ('OrganS', 0.02, '57.2±4.7', '39.5±2.0', '42.5±1.4', '52.3±1.3', '63.3±1.2', '59.6±2.7', '63.2±0.6', '63.7±0.7'),
    ('OrganS', 0.05, '66.0±1.2', '51.4±0.9', '56.6±1.1', '66.0±0.9', '67.7±1.1', '66.0±0.8', '68.1±0.7', '68.4±1.0'),
    ('OrganA', 0.02, '83.6±2.1', '63.1±0.6', '72.9±1.8', '68.6±2.2', '84.9±1.1', '83.3±1.5', '86.3±0.7', '86.5±0.9'),
    ('OrganA', 0.05, '89.8±1.2', '80.9±0.8', '84.6±0.2', '81.0±2.0', '90.5±0.9', '90.7±0.6', '90.4±0.7', '91.9±0.3'),
    ('Path', 0.02, '77.5±4.6', '36.8±1.0', '53.5±3.9', '53.5±5.7', '77.0±1.6', '73.0±3.8', '78.0±2.1', '80.9±0.9'),
    ('Path', 0.05, '84.0±2.4', '49.9±2.4', '58.9±4.2', '58.7±3.5', '82.5±2.2', '83.0±1.6', '84.6±2.7', '85.9±1.7'),
    ('Tissue', 0.02, '43.1±1.0', '10.1±0.4', '39.3±0.6', '39.8±0.6', '44.7±1.0', '35.6±0.9', '42.9±0.9', '42.6±0.6'),
    ('Tissue', 0.05, '49.1±0.4', '15.2±0.2', '43.6±1.0', '46.7±1.0', '48.1±0.6', '41.0±1.1', '48.3±0.4', '49.5±1.3'),
    ('Blood', 0.02, '83.2±1.5', '48.3±6.9', '67.5±3.3', '75.9±2.5', '82.7±1.6', '78.8±2.1', '83.1±3.0', '84.5±1.7'),
    ('Blood', 0.05, '90.3±0.5', '72.5±1.7', '81.4±2.1', '86.6±1.7', '88.2±2.7', '89.0±2.5', '92.3±0.6', '93.4±0.7'),
]
paper = pd.DataFrame(paper_rows, columns=['dataset','ratio','Random','EL2N','Forgetting','EVA','Facility','FPS','Herding','Graph-A2/Ours'])
display(paper)

## 3. Graph-A2 seed-42 test 复现

下面是已经完成且审计通过的 10/10 个 test runs。`best_ba` 是作者代码每 10 epochs 评估后记录的最大值，是 Table 1 的主比较指标；`final_ba` 是第 1000 epoch 的诊断结果。注意：原作者运行时记录 best 分数和 epoch，但没有持久化对应模型权重。

In [ ]:
a2_snapshot = pd.DataFrame([
    ('OrganS',0.02,42,63.0122,63.1239,970,63.7,0.7),
    ('OrganS',0.05,42,67.6700,67.7612,950,68.4,1.0),
    ('OrganA',0.02,42,84.2337,84.3789,850,86.5,0.9),
    ('OrganA',0.05,42,91.4012,91.9195,400,91.9,0.3),
    ('Path',0.02,42,81.5916,82.7834,240,80.9,0.9),
    ('Path',0.05,42,85.7944,87.6756,380,85.9,1.7),
    ('Tissue',0.02,42,33.9159,42.1597,70,42.6,0.6),
    ('Tissue',0.05,42,44.1026,48.2169,170,49.5,1.3),
    ('Blood',0.02,42,82.0366,82.1497,900,84.5,1.7),
    ('Blood',0.05,42,93.2140,93.3013,900,93.4,0.7),
], columns=['dataset','ratio','seed','final_ba','best_ba','best_epoch','paper_mean','paper_std'])

a2_live = read_result_jsons(A2_ROOT, 'test_result.json')
if not a2_live.empty:
    a2_live = a2_live.rename(columns={'balanced_accuracy':'final_ba','best_balanced_accuracy':'best_ba','training_seed':'seed'})
    for col in ['final_ba','best_ba']:
        a2_live[col] *= 100
    a2_current = a2_live.merge(a2_snapshot[['dataset','ratio','paper_mean','paper_std']], on=['dataset','ratio'], how='left')
else:
    a2_current = a2_snapshot.copy()

a2_seed42 = a2_current[a2_current['seed'] == 42].copy()
a2_seed42['delta_final_pp'] = a2_seed42['final_ba'] - a2_seed42['paper_mean']
a2_seed42['delta_best_pp'] = a2_seed42['best_ba'] - a2_seed42['paper_mean']
display(a2_seed42[['dataset','ratio','final_ba','best_ba','best_epoch','paper_mean','paper_std','delta_final_pp','delta_best_pp']].round(2))

In [ ]:
import matplotlib.pyplot as plt
plot_df = a2_seed42.copy()
plot_df['condition'] = plot_df['dataset'] + ' ' + (plot_df['ratio'] * 100).astype(int).astype(str) + '%'
ax = plot_df.set_index('condition')[['paper_mean','final_ba','best_ba']].plot(kind='bar', figsize=(13,5), width=0.8)
ax.set_ylabel('Balanced accuracy (%)')
ax.set_title('Graph-A2 seed 42: paper mean vs reproduced final/best checkpoint')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

In [ ]:
if a2_live.empty:
    print('本地没有实时 Graph-A2 JSON；静态快照目前只有 seed 42。')
else:
    a2_multi = (a2_live.groupby(['dataset','ratio'])
        .agg(n_seeds=('seed','nunique'), seeds=('seed', lambda x: ','.join(map(str, sorted(set(x))))),
             final_ba_mean=('final_ba','mean'), final_ba_std=('final_ba','std'),
             best_ba_mean=('best_ba','mean'), best_ba_std=('best_ba','std'))
        .reset_index())
    print(f'Graph-A2 test results: {len(a2_live)}/30（目标为三种子）')
    display(a2_multi.round(2))

### Graph-A2 结果分析

- **PathMNIST**：best BA 为 82.78%/87.68%，相对论文 80.9%/85.9% 分别高 1.88/1.78 pp。
- **OrganS/OrganA/Blood**：多数条件距论文均值约 0.2–2.5 pp，单种子落在或接近论文五种子波动范围，必须等待三种子均值再判断。
- **TissueMNIST**：best BA 为 42.16%/48.22%，接近论文 42.6%/49.5%；final BA 仅 33.92%/44.10%，说明 final epoch 不适合作为该表主口径。
- 作者 GitHub 在训练期间每 10 epochs 读取 evaluation split 并记录 test-best 分数与 epoch，但不保存对应 checkpoint。为复现 Table 1，本报告以 best BA 为主；final BA 与 final per-class 指标仅作诊断，同时明确记录 test leakage 风险。

## 4. v11 seed-42、5% validation 结果

候选含义：`a3_more_hops` 增加扩散深度；`a4_larger_k` 增大邻域；`a5_full_ppr` 联合增大 k、扩散深度和几何衰减；`a6_full_ppr_cap1` 在 a5 上加入与 A0 相同的 unsafe-sample cap。

In [ ]:
v11_seed42_rows = [
('OrganS','a0_original',77.9438,78.8648,52.6316),('OrganS','a3_more_hops',77.3370,77.7918,32.8571),('OrganS','a4_larger_k',78.7648,79.2288,29.4737),('OrganS','a5_full_ppr',77.3671,78.0730,31.5789),('OrganS','a6_full_ppr_cap1',78.0103,78.3778,33.6842),
('OrganA','a0_original',97.5799,97.9279,88.4688),('OrganA','a3_more_hops',97.1403,98.0166,86.5784),('OrganA','a4_larger_k',97.7967,98.5439,87.3346),('OrganA','a5_full_ppr',98.1525,98.2711,94.1915),('OrganA','a6_full_ppr_cap1',97.9223,98.2329,92.4386),
('Path','a0_original',96.0422,96.4565,87.7512),('Path','a3_more_hops',94.7506,94.8008,86.0243),('Path','a4_larger_k',96.6675,96.8877,92.0574),('Path','a5_full_ppr',95.8032,96.4268,89.2823),('Path','a6_full_ppr_cap1',95.9496,96.4455,89.4737),
('Tissue','a0_original',44.3634,47.9006,35.0590),('Tissue','a3_more_hops',45.1683,48.5305,39.1924),('Tissue','a4_larger_k',45.4621,47.8226,35.6948),('Tissue','a5_full_ppr',45.4867,48.2081,37.0572),('Tissue','a6_full_ppr_cap1',44.7980,48.1108,37.3321),
('Blood','a0_original',94.3008,94.7809,73.4483),('Blood','a3_more_hops',92.7438,93.1104,70.0000),('Blood','a4_larger_k',93.3079,93.3459,70.6897),('Blood','a5_full_ppr',89.5877,89.8957,58.6207),('Blood','a6_full_ppr_cap1',93.5118,93.6986,75.1724),
]
v11_seed42 = pd.DataFrame(v11_seed42_rows, columns=['dataset','variant','final_ba','best_ba','worst_recall'])
reference = v11_seed42[v11_seed42.variant == 'a0_original'].set_index('dataset')
v11_seed42['gain_vs_a0_pp'] = v11_seed42.apply(lambda row: row.best_ba - reference.loc[row.dataset, 'best_ba'], axis=1)
v11_seed42['worst_delta_pp'] = v11_seed42.apply(lambda row: row.worst_recall - reference.loc[row.dataset, 'worst_recall'], axis=1)
v11_seed42['passes_seed42_gate'] = (v11_seed42.gain_vs_a0_pp >= 0.5) & (v11_seed42.worst_delta_pp >= -2.0)
display(v11_seed42.sort_values(['dataset','best_ba'], ascending=[True,False]).round(2))

In [ ]:
seed42_candidates = v11_seed42[(v11_seed42.variant != 'a0_original') & v11_seed42.passes_seed42_gate]
seed42_winners = (seed42_candidates.sort_values(['dataset','gain_vs_a0_pp']).groupby('dataset').tail(1))
fallback = sorted(set(v11_seed42.dataset) - set(seed42_winners.dataset))
display(seed42_winners[['dataset','variant','best_ba','gain_vs_a0_pp','worst_delta_pp']].round(2))
print('Seed-42 下没有通过门槛、暂时保留 A0 的数据集：', fallback)

### v11 单种子分析

- OrganA：`a5_full_ppr` 相对 A0 提升约 0.57 pp，worst recall 同时提高。
- Path：`a4_larger_k` 提升约 0.63 pp，worst recall 提高约 4.31 pp。
- Tissue：多个候选提高 BA；`a5_full_ppr` 单种子提升约 1.12 pp。
- OrganS：`a4_larger_k` 虽提高 BA，但 worst recall 从 52.63% 降至 29.47%，不应接受。
- Blood：所有候选 BA 都低于 A0，应保留 A0。

这些只是 seed-42 观察，不能直接冻结。正式 winner 由 seed 42/43/44 的 validation mean 决定。

## 5. 三种子 validation 动态进展

最后一次已确认快照为 **55/75**。运行下面单元格会读取当前目录下所有 `val_result.json`，按数据集和候选显示已有种子数、均值和标准差。只有每组 `n_seeds=3` 且总数达到 75 才允许 freeze。

In [ ]:
v11_live = read_result_jsons(V11_VAL_ROOT, 'val_result.json')
if v11_live.empty:
    print('本地没有 HPC validation JSON。静态 seed-42 结果见上一节。')
else:
    summary = (v11_live.groupby(['dataset','ratio','variant'])
        .agg(n_seeds=('training_seed','nunique'), seeds=('training_seed', lambda x: ','.join(map(str, sorted(set(x))))),
             best_ba_mean=('best_balanced_accuracy','mean'), best_ba_std=('best_balanced_accuracy','std'),
             final_ba_mean=('balanced_accuracy','mean'), final_ba_std=('balanced_accuracy','std'),
             worst_mean=('worst_class_recall','mean'), cvar20_mean=('class_cvar20','mean'))
        .reset_index())
    for col in ['best_ba_mean','best_ba_std','final_ba_mean','final_ba_std','worst_mean','cvar20_mean']:
        summary[col] *= 100
    print(f'Validation results: {len(v11_live)}/75')
    display(summary.sort_values(['dataset','best_ba_mean'], ascending=[True,False]).round(2))

## 6. Selection 完整性审计

预期输出为 100 组：5 数据集 × 2 比例 × 10 variants。其中 10 组 A0 必须与隔离 Table 1 的 Graph-A2 文件逐字节一致；v11 只生成其余 90 组改进候选。

In [ ]:
datasets = ['organsmnist','organamnist','pathmnist','tissuemnist','bloodmnist']
ratios = ['r0p02','r0p05']
metrics_count = len(list(V11_SELECTION_ROOT.rglob('selection_metrics.json'))) if V11_SELECTION_ROOT.exists() else 0
exact = []
for dataset in datasets:
    for ratio in ratios:
        official = PROJECT / f'table1_reproduction/outputs/job1_selection/{dataset}/{ratio}/graph_a2/seed42/selected_indices.npy'
        copied = V11_SELECTION_ROOT / f'{dataset}/{ratio}/a0_original/selected_indices.npy'
        exact.append(official.exists() and copied.exists() and official.read_bytes() == copied.read_bytes())
print(f'selection metrics: {metrics_count}/100')
print(f'exact frozen A0 matches: {sum(exact)}/10')

## 7. Slurm 任务链与实时状态

| Job | 作用 | 依赖 | 预期输出 |
|---|---|---|---|
| 26685 | v11 三种子、5% validation | 无 | 75 个 val results |
| 26690 | 作者 Graph-A2 三种子 test | 等并发槽位 | 30 个 test results，已有 seed 42 的 10 个会跳过 |
| 26691 | 冻结每个数据集 winner | `afterok:26685` | test config + freeze manifest |
| 26703 | A0 与冻结 v11 winner，五种子 test | `afterok:26691` | 最终 test JSON |

freeze gate：mean best validation BA 至少比 A0 高 0.5 pp，并且单独标注的 final-epoch mean worst-class recall 最多下降 2 pp。

In [ ]:
try:
    status = subprocess.run(
        ['squeue','-j','26685,26690,26691,26703','-o','%i|%j|%T|%M|%R'],
        check=False, capture_output=True, text=True, timeout=15
    )
    print(status.stdout or status.stderr)
except (FileNotFoundError, subprocess.TimeoutExpired):
    print('当前环境没有可用的 Slurm 客户端；请在 HPC 上运行此单元格。')

## 8. 冻结 winner 与最终 test（完成后自动显示）

In [ ]:
manifest_path = FROZEN_ROOT / 'job2_table1_test.freeze_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    decision_rows = []
    for dataset, item in manifest['decisions'].items():
        decision_rows.append({
            'dataset': dataset, 'reference': item['reference'], 'winner': item['winner'],
            'validation_best_ba_gain_pp': 100 * item['winner_best_balanced_accuracy_gain']
        })
    display(pd.DataFrame(decision_rows).sort_values('dataset').round(3))
else:
    print('TODO: validation 未完成，freeze manifest 尚未生成。')

final_test = read_result_jsons(V11_TEST_ROOT, 'test_result.json')
if not final_test.empty:
    final_summary = (final_test.groupby(['dataset','ratio','variant'])
        .agg(n_seeds=('training_seed','nunique'), best_ba_mean=('best_balanced_accuracy','mean'),
             best_ba_std=('best_balanced_accuracy','std'), final_ba_mean=('balanced_accuracy','mean'),
             final_ba_std=('balanced_accuracy','std'), final_worst_mean=('worst_class_recall','mean'))
        .reset_index())
    for col in ['best_ba_mean','best_ba_std','final_ba_mean','final_ba_std','final_worst_mean']:
        final_summary[col] *= 100
    print('Final test result files:', len(final_test))
    display(final_summary.round(2))
else:
    print('TODO: 最终五种子 test 尚未产生结果。')

## 9. 结果解释边界

- 当前 Graph-A2 seed-42 test 是复现结果，不是五种子最终结论。
- 当前 v11 seed-42 表是 validation 探索结果，不可与论文 test 数字直接比较。
- v11 三种子 validation 只能用于冻结算法变体，不用于论文最终性能报告。
- 冻结后，A0 与 v11 使用完全相同的五个训练种子和 test evaluator，才能做公平结论。
- Table 1 和 v11 主比较统一使用 best BA；final BA 与 final per-class 指标必须分开标注。作者式 test-best 选择存在 test leakage 风险，尤其会影响 TissueMNIST。
- 五个数据集分别选择不同参数可以作为 calibration protocol，但论文中必须明确这是 validation-selected dataset-specific configuration，不能称为单一无参数算法。

## 10. TODO

### 正在运行

- [ ] Job 26685：完成 v11 3-seed validation，达到 75/75。
- [ ] Job 26690：完成 Graph-A2 seeds 42/43/44 test，达到 30/30。
- [ ] Job 26691：根据固定 gate 生成 winner manifest；人工检查每个数据集的 mean BA、worst recall 和种子完整性。
- [ ] Job 26703：完成冻结 A0/v11 的 seeds 42–46、2%/5% test。

### 结果汇总

- [ ] 汇总 Graph-A2 三种子 best BA mean±std，并与论文五种子 mean±std 比较。
- [ ] 汇总最终 A0 与 v11 五种子 best BA；final BA、final worst recall、final CVaR20 单列诊断。
- [ ] 对同 seed 的 v11−A0 BA 做 paired difference、95% CI；五个种子样本很小，不夸大显著性。
- [ ] 单独分析 TissueMNIST 的 final-vs-best gap、训练曲线和过拟合。
- [ ] 检查所有 result JSON 的 selection hash、config、split 和 `test_read` 字段。

### 论文与工程记录

- [ ] 冻结最终 commit、环境版本、数据 MD5、UNI 权重来源和 Slurm Job IDs。
- [ ] 导出最终 CSV/Markdown 表格和训练成本统计。
- [ ] 明确声明作者 GitHub 的 test-every-10-epochs 行为、本研究采用的 best-epoch BA，以及未持久化 best checkpoint 的限制。
- [ ] 在独立服务器复核 notebook 全部单元格可重现。

## 11. 关键文件与输出地址

```text
/project/prj-sis01/xuxiaoyu/graph_bench/
├── table1_reproduction/
│   ├── vendor/                         # 作者 8cf757a 隔离实现
│   ├── outputs/job1_selection/         # 冻结 selection
│   └── outputs/job2_table1/            # Graph-A2 test results
├── v11/
│   ├── outputs/job1_table1_calibration/ # 100 组 selection
│   ├── outputs/job2_table1_validation/  # 3-seed validation
│   ├── outputs/frozen_protocol/         # winner manifest + test config
│   └── outputs/job2_table1_test/        # 最终 5-seed test
└── reports/graph_a2_v11_experiment_report.ipynb
```

Slurm 日志位于 `table1_reproduction/logs/` 和 `v11/logs/`。